In [1]:
import numpy as np  # import numerical python
import matplotlib.pyplot as plt  # import plotting functions
import seaborn as sns  # import nicer plotting functions
import polars as pl  # import polars to import data
import tifffile as tiff
from tifffile import imwrite, imread
from copy import deepcopy
import os
import time
from copy import copy
from skimage.filters import gaussian
import xarray as xr
import sys

sys.path.append("..")

from src import IOFunctions

IO = IOFunctions.IO_Functions()

from src import Multicolour_Simulation_Functions

MSF = Multicolour_Simulation_Functions.MultiC_Sim_Funcs()

from src import PSFFunctions

PSF = PSFFunctions.PSF_Functions()

from src import PlottingFunctions

plotter = PlottingFunctions.Plotter()

from src import ImageAnalysisFunctions

I_AF = ImageAnalysisFunctions.Image_Analysis_Functions()

from src import sCMOSFunctions

sCMOS = sCMOSFunctions.sCMOS_Functions()

from scipy.ndimage import gaussian_filter

from src import SpectralFunctions

S_F = SpectralFunctions.Spectral_Funcs()

from src import MaskFunctions

MaskF = MaskFunctions.Mask_Functions()

In [2]:
data_folder = r"C:\Users\jsb92\Cambridge University Dropbox\Joseph Beckwith\Chemistry\Lee\Data\Salix\CS505CU_Calibration"
# data_folder = "/home/jbeckwith/Documents/Cambridge University Dropbox/Joseph Beckwith/Chemistry/Lee/Data/Salix/CS505CU_Calibration"
gain = gaussian_filter(IO.read_tiff(os.path.join(data_folder, "gain.tif")), 4)
offset = gaussian_filter(IO.read_tiff(os.path.join(data_folder, "offset.tif")), 4)
variance = gaussian_filter(IO.read_tiff(os.path.join(data_folder, "variance.tif")), 4)
readnoise = gaussian_filter(IO.read_tiff(os.path.join(data_folder, "readnoise.tif")), 4)
rqe = gaussian_filter(IO.read_tiff(os.path.join(data_folder, "rqe.tif")), 4)

In [3]:
R, G, B, wavelength = S_F.getpixelefficiency()

# B[wavelength < 500] = 0.8
# B[wavelength >= 500] = 0
# G[(wavelength < 500) | (wavelength > 600)] = 0
# G[(wavelength < 600) & (wavelength >= 500)] = 0.8
# R[wavelength < 600] = 0
# R[wavelength >= 600] = 0.8
# R = 0.8*((R+G+B)/np.max(R+G+B))
# G = R
# B = R
wavelength = wavelength
pixel_QYs = np.vstack([B, G, R])

background_photons = 0
NA = 1.49

In [4]:
import types

fitter = types.SimpleNamespace()
fitter.default_params = np.array(["xc", "yc", "A", "sigma", "b", "B", "G", "R"])
fitter.fit_function = I_AF.WLS_fit_colour
fitter.error_type = "Smoothed"

In [5]:
n_photon = 1000
n_bootstrap = 10000
background_photons = 0
pixel_scan = 2
pixel_size_lspace = np.arange(20, 200 + pixel_scan, pixel_scan)
NA = 1.49

In [6]:
camera_calibration = {}
camera_calibration["gain"] = gain
camera_calibration["offset"] = np.zeros_like(offset) + 100
camera_calibration["variance"] = variance
camera_calibration["readnoise"] = readnoise
camera_calibration["rqe"] = rqe
camera_calibration["pixel_QYs"] = pixel_QYs
camera_calibration["pixel_order"] = ["B", "G", "R"]
camera_calibration["masks"] = MaskF.get_masks(
    mosaic_unit=np.array([["B", "G"], ["G", "R"]]),
    size_x=variance.shape[0],
    size_y=variance.shape[1],
)
camera_calibration["pixel_order_indices"] = np.arange(3)

In [7]:
dyes = ["ATTO 488", "ATTO 550", "ATTO 647N", "ATTO 740"]
filters = []

In [8]:
save_folder = r"C:\Users\jsb92\Cambridge University Dropbox\Joseph Beckwith\Chemistry\Lee\Data\Simulation\20250212_PixelSizeTest"

In [ ]:
for dye in dyes:
    MSF.test_pixel_size(
        dye,
        filters,
        wavelength,
        camera_calibration,
        fitter,
        save_folder,
        pixel_size_lspace,
        starting_flag="fitter_",
        n_bootstrap=n_bootstrap,
        background_photons=background_photons,
        NA=1.49,
        n_photon=n_photon,
        cpu_fraction=0.9,
        physical_image_size=1500.0,
    )